加载数据


In [19]:
import pandas as pd

raw_path = '/UK_Smart_Retail_Data_Ops/original_data/'
customers = pd.read_csv(raw_path + 'olist_customers_dataset.csv')
geolocation = pd.read_csv(raw_path + 'olist_geolocation_dataset.csv')
order_items = pd.read_csv(raw_path + 'olist_order_items_dataset.csv')
order_payments = pd.read_csv(raw_path + 'olist_order_payments_dataset.csv')
order_reviews = pd.read_csv(raw_path + 'olist_order_reviews_dataset.csv')
orders = pd.read_csv(raw_path + 'olist_orders_dataset.csv')
products = pd.read_csv(raw_path + 'olist_products_dataset.csv')
sellers = pd.read_csv(raw_path + 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(raw_path + 'product_category_name_translation.csv')

print("所有文件加载成功！")

所有文件加载成功！


In [20]:
# 1. 查看每个表的行数和列数
print("========== 各表维度 ==========")
print(f"customers: {customers.shape}")
print(f"geolocation: {geolocation.shape}")
print(f"order_items: {order_items.shape}")
print(f"order_payments: {order_payments.shape}")
print(f"order_reviews: {order_reviews.shape}")
print(f"orders: {orders.shape}")
print(f"products: {products.shape}")
print(f"sellers: {sellers.shape}")
print(f"category_translation: {category_translation.shape}")


========== 各表维度 ==========
customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [21]:
# 2. 查看orders的列名和数据类型，以及缺失值数量
print("\n========== orders 表信息 ==========")
print(orders.info())



========== orders 表信息 ==========
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
None


In [22]:
# 3. 重点检查 orders 表中的送达日期缺失情况
missing_delivery = orders['order_delivered_customer_date'].isna().sum()
total_orders = len(orders)
print(f"orders 表总订单数：{total_orders}")
print(f"缺失实际送达日期的订单数：{missing_delivery}")
print(f"缺失占比：{missing_delivery/total_orders:.2%}")

orders 表总订单数：99441
缺失实际送达日期的订单数：2965
缺失占比：2.98%


将订单相关的时间列从字符串转为 Pandas 的 datetime 类型，便于后续按时间筛选、计算时间差等。


In [23]:
# 1. orders 表的多列日期转换
date_cols_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols_orders:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# 2. order_items 表的日期
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')

# 3. order_reviews 表的日期
order_reviews['review_creation_date'] = pd.to_datetime(order_reviews['review_creation_date'], errors='coerce')
order_reviews['review_answer_timestamp'] = pd.to_datetime(order_reviews['review_answer_timestamp'], errors='coerce')

# 4. 验证转换结果（查看 orders 表前几行的日期列）
print("orders 表日期列转换后：")
print(orders[date_cols_orders].head())

orders 表日期列转换后：
  order_purchase_timestamp   order_approved_at order_delivered_carrier_date  \
0      2017-10-02 10:56:33 2017-10-02 11:07:15          2017-10-04 19:55:00   
1      2018-07-24 20:41:37 2018-07-26 03:24:27          2018-07-26 14:31:00   
2      2018-08-08 08:38:49 2018-08-08 08:55:23          2018-08-08 13:50:00   
3      2017-11-18 19:28:06 2017-11-18 19:45:59          2017-11-22 13:39:59   
4      2018-02-13 21:18:39 2018-02-13 22:20:29          2018-02-14 19:46:34   

  order_delivered_customer_date order_estimated_delivery_date  
0           2017-10-10 21:25:13                    2017-10-18  
1           2018-08-07 15:27:45                    2018-08-13  
2           2018-08-17 18:06:29                    2018-09-04  
3           2017-12-02 00:28:42                    2017-12-15  
4           2018-02-16 18:17:02                    2018-02-26  


products 表中的 product_category_name 是葡萄牙语，我们需要借助 category_translation 表将其翻译为英文

In [24]:
# 1. 查看翻译表的前几行
print("翻译表示例：")
print(category_translation.head())
# 2. 将 products 表与翻译表进行左连接
products = products.merge(category_translation, on='product_category_name', how='left')
# 3. 检查合并后的效果
print("合并后 products 表（前5行）：")
print(products[['product_category_name', 'product_category_name_english']].head(10))
# 4. 查看哪些产品类别没有翻译成功（即英文名为空）
missing_translation = products['product_category_name_english'].isna().sum()
total_products = len(products)
print(f"产品总数：{total_products}")
print(f"缺失英文类别的产品数：{missing_translation}")
print(f"缺失占比：{missing_translation/total_products:.2%}")


翻译表示例：
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor
合并后 products 表（前5行）：
   product_category_name product_category_name_english
0             perfumaria                     perfumery
1                  artes                           art
2          esporte_lazer                sports_leisure
3                  bebes                          baby
4  utilidades_domesticas                    housewares
5  instrumentos_musicais           musical_instruments
6             cool_stuff                    cool_stuff
7       moveis_decoracao               furniture_decor
8       eletrodomesticos               home_appliances
9             brinquedos                          toys
产品总数：32951
缺失英文类别的产品数：623
缺失占比：

In [25]:
# 调查623个缺失英文类别的产品
missing_cat = products[products['product_category_name_english'].isna()]
print(f"缺失英文类别的产品数：{len(missing_cat)}")
print("\n葡语类别名分布：")
print(missing_cat['product_category_name'].value_counts(dropna=False))

缺失英文类别的产品数：623

葡语类别名分布：
product_category_name
NaN                                              610
portateis_cozinha_e_preparadores_de_alimentos     10
pc_gamer                                           3
Name: count, dtype: int64


In [26]:
# products 表全列缺失值统计
print("========== products 表缺失值统计 ==========")
print(products.isnull().sum())

========== products 表缺失值统计 ==========
product_id                         0
product_category_name            610
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english    623
dtype: int64


In [27]:
# 澄清 geolocation 重复逻辑
print("geolocation 总行数：", len(geolocation))
print("唯一邮编前缀数：", geolocation['geolocation_zip_code_prefix'].nunique())
print("完全重复的行数：", geolocation.duplicated().sum())

geolocation 总行数： 1000163
唯一邮编前缀数： 19015
完全重复的行数： 261831


In [28]:
# Task 1: 删除 product_category_name 为空的僵尸产品
# .notna() 保留非空行，相当于"只要有类别名的产品"
before = len(products)
products = products[products['product_category_name'].notna()].copy()
after = len(products)

print(f"删除前：{before} 行")
print(f"删除后：{after} 行")
print(f"共删除：{before - after} 行")
print(f"期望删除：610 行 → {'✅ 正确' if before - after == 610 else '❌ 异常'}")

删除前：32951 行
删除后：32341 行
共删除：610 行
期望删除：610 行 → ✅ 正确


In [29]:
# Task 2: 手动补充翻译表里缺失的两个类别
# pc_gamer 和 portateis_cozinha_e_preparadores_de_alimentos 在翻译表里没有收录
# 用 .loc 条件定位 + 直接赋值

products.loc[
    products['product_category_name'] == 'pc_gamer',
    'product_category_name_english'
] = 'pc_gamer'

products.loc[
    products['product_category_name'] == 'portateis_cozinha_e_preparadores_de_alimentos',
    'product_category_name_english'
] = 'portable_kitchen_food_preparers'

# 验证：现在英文类别还剩几个空？应该是 0
remaining = products['product_category_name_english'].isna().sum()
print(f"补充后仍缺失英文类别的产品数：{remaining}")
print(f"期望结果：0 → {'✅ 正确' if remaining == 0 else '❌ 异常'}")

补充后仍缺失英文类别的产品数：0
期望结果：0 → ✅ 正确


In [30]:
# Task 3: 用中位数填充尺寸/重量的缺失值
# 为什么用中位数而不是平均数？
# 因为商品重量/尺寸可能有极端值（比如超重货物），中位数更稳健

cols_to_fill = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in cols_to_fill:
    median_val = products[col].median()
    products[col] = products[col].fillna(median_val)
    print(f"{col} → 用中位数 {median_val} 填充")

# 验证：这4列应该全部为0缺失
print(f"\n填充后缺失值：")
print(products[cols_to_fill].isnull().sum())

product_weight_g → 用中位数 700.0 填充
product_length_cm → 用中位数 25.0 填充
product_height_cm → 用中位数 13.0 填充
product_width_cm → 用中位数 20.0 填充

填充后缺失值：
product_weight_g     0
product_length_cm    0
product_height_cm    0
product_width_cm     0
dtype: int64


In [31]:
# Task 4: 去除 geolocation 完全重复的行
# drop_duplicates() 默认保留第一次出现的行，删除后续重复的
before = len(geolocation)
geolocation = geolocation.drop_duplicates().copy()
after = len(geolocation)

print(f"去重前：{before} 行")
print(f"去重后：{after} 行")
print(f"共删除：{before - after} 行")
print(f"期望删除：261831 行 → {'✅ 正确' if before - after == 261831 else '❌ 异常'}")

去重前：1000163 行
去重后：738332 行
共删除：261831 行
期望删除：261831 行 → ✅ 正确


In [33]:
# 保存清洗后的数据
output_path = "C:/Users/许佳佳/Desktop/UK_Smart_Retail_Data_Ops/cleaned_"

customers.to_csv(output_path + 'customers.csv', index=False)
geolocation.to_csv(output_path + 'geolocation.csv', index=False)
order_items.to_csv(output_path + 'order_items.csv', index=False)
order_payments.to_csv(output_path + 'order_payments.csv', index=False)
order_reviews.to_csv(output_path + 'order_reviews.csv', index=False)
orders.to_csv(output_path + 'orders.csv', index=False)
products.to_csv(output_path + 'products.csv', index=False)
sellers.to_csv(output_path + 'sellers.csv', index=False)

print("✅ 所有清洗后的文件已保存到 cleaned_data 文件夹")

✅ 所有清洗后的文件已保存到 cleaned_data 文件夹
